# Phase 2 — Preprocessing
Rainfall-triggered landslide detection pipeline

Scope: cloud/shadow masking, inventory cleaning, and the joint spatio-temporal negative sampling loop. Builds directly on Phase 1's raw pulls — this notebook re-declares the handful of Phase 1 objects it actually needs (ROI, SRTM/slope, the S2+cloud-probability join) rather than re-running all of Phase 1, since GEE objects don't persist across notebook sessions.

**Output of this notebook:** a cleaned, labeled, spatially-blocked point dataset (positives + negatives) ready for Phase 3 feature extraction.

## Setup

In [ ]:
import ee
import geemap.core as geemap
import pandas as pd
import numpy as np

ee.Authenticate()
ee.Initialize(project='landslidedetection-506008')  # <-- your GEE-linked Cloud project
print("Earth Engine initialized:", ee.Number(1).add(1).getInfo() == 2)

# scikit-gstat is needed for the empirical semivariogram check later
!pip install -q scikit-gstat pyproj
import skgstat as skg
import pyproj

print("Ready.")


## Re-declare what Phase 1 already validated
Same ROI and date ranges as Phase 1, so this stays consistent with what was actually ingested. Only pulling what Phase 2 needs directly: SRTM/slope (for the stable-terrain mask), the S2+cloud-probability join (for the masking function), and JRC Global Surface Water (permanent water mask).

In [ ]:
lat_min, lat_max = 26.0, 35.0
lon_min, lon_max = 74.0, 97.0
roi = ee.Geometry.Rectangle([lon_min, lat_min, lon_max, lat_max])

# Kept identical to Phase 1 so the inventory date filter below stays consistent
# with what was actually ingested for Sentinel-1/2 (not CHIRPS's longer 2014 start).
S2_START, S2_END = "2017-01-01", "2023-12-31"

srtm = ee.Image("USGS/SRTMGL1_003").select("elevation").clip(roi)
srtm_slope = ee.Terrain.slope(srtm)

s2_sr = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
         .filterBounds(roi).filterDate(S2_START, S2_END))
s2_cloud_prob = (ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")
                  .filterBounds(roi).filterDate(S2_START, S2_END))
s2_join = ee.Join.saveFirst('s2cloudless').apply(
    primary=s2_sr, secondary=s2_cloud_prob,
    condition=ee.Filter.equals(leftField='system:index', rightField='system:index'))
s2_with_prob = ee.ImageCollection(s2_join)

water_occurrence = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence").unmask(0)

print("Re-declared ROI, SRTM/slope, S2+cloud-prob join, and water occurrence layer.")


## 1. Cloud + shadow masking
`s2cloudless` alone misses cloud shadows (they'd still pass a cloud-probability filter and quietly corrupt NDVI/NDWI). This combines the cloud-probability mask with the SCL shadow class and a spectral dark-pixel check, per the technical review earlier.

In [ ]:
def mask_s2_clouds_and_shadows(image, max_cloud_prob=20):
    """Masks clouds (s2cloudless) and cloud shadows (SCL + NIR/SWIR dark-pixel check)."""
    cloud_prob = ee.Image(image.get('s2cloudless')).select('probability')
    is_cloud = cloud_prob.gte(max_cloud_prob)

    # SCL class 3 = cloud shadow
    scl = image.select('SCL')
    is_shadow_scl = scl.eq(3)

    # Spectral shadow confirmation: dark NIR/SWIR pixels near a detected cloud
    dark_pixels = image.select('B8').lt(1500).And(image.select('B11').lt(1000))
    shadow_near_cloud = is_cloud.focal_max(radius=60, units='meters').And(dark_pixels)

    combined_mask = is_cloud.Or(is_shadow_scl).Or(shadow_near_cloud)

    optical_bands = image.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12']).divide(10000)
    masked_image = optical_bands.updateMask(combined_mask.Not()).copyProperties(image, ['system:time_start'])
    return ee.Image(masked_image)  # copyProperties() returns a generic ee.Element, cast back to ee.Image


# Quick test: compare masked-fraction on a single scene before/after
test_img = s2_with_prob.first()
masked = mask_s2_clouds_and_shadows(test_img)

total_px = test_img.select('B4').reduceRegion(
    reducer=ee.Reducer.count(), geometry=roi, scale=100, maxPixels=1e9, bestEffort=True
).getInfo()
valid_px_after = masked.select('B4').reduceRegion(
    reducer=ee.Reducer.count(), geometry=roi, scale=100, maxPixels=1e9, bestEffort=True
).getInfo()

print("Pixels before masking (coarse check):", total_px)
print("Valid pixels after cloud+shadow masking (coarse check):", valid_px_after)


## 2. Inventory cleaning
Applied as an explicit funnel so every drop is visible and auditable, rather than one opaque filter chain. Filters, in order: parse dates → restrict to ROI → restrict to the date range actually covered by the ingested Sentinel-1/2 imagery → keep only rainfall-related triggers (rainfall-only scope) → keep only positionally-accurate points (≤ 5 km) → deduplicate points landing in the same ~pixel/day.

In [ ]:
df_raw = pd.read_csv("Landslide_Reports.csv")
print(f"0. Raw rows: {len(df_raw)}")

# Parse dates
df_raw["event_date_parsed"] = pd.to_datetime(
    df_raw["Event Date"], format="%d/%m/%Y, %H:%M", errors="coerce"
)
step1 = df_raw[df_raw["event_date_parsed"].notna()].copy()
print(f"1. With a parseable date: {len(step1)}")

# Restrict to ROI
step2 = step1[
    step1["Latitude"].between(lat_min, lat_max) &
    step1["Longitude"].between(lon_min, lon_max)
].copy()
print(f"2. Inside ROI: {len(step2)}")

# Restrict to the date range Sentinel-1/2 were actually ingested for in Phase 1
step3 = step2[
    step2["event_date_parsed"].between(pd.Timestamp(S2_START), pd.Timestamp(S2_END))
].copy()
print(f"3. Inside ingested date range ({S2_START} to {S2_END}): {len(step3)}")

# Rainfall-only scope
rainfall_triggers = ["Downpour", "Rain", "Continuous rain", "Monsoon", "Tropical cyclone"]
step4 = step3[step3["Landslide Trigger"].isin(rainfall_triggers)].copy()
print(f"4. Rainfall-related trigger: {len(step4)}")

# Positional accuracy <= 5 km
accurate_labels = ["Known exactly", "Known within 1 km", "Known within 5 km"]
step5 = step4[step4["Location Accuracy"].isin(accurate_labels)].copy()
print(f"5. Location accuracy <= 5 km: {len(step5)}")

# Deduplicate: same ~pixel (rounded to ~0.003 deg, ~300m) and same day
step5["lat_round"] = step5["Latitude"].round(3)
step5["lon_round"] = step5["Longitude"].round(3)
step5["date_round"] = step5["event_date_parsed"].dt.date
step6 = step5.drop_duplicates(subset=["lat_round", "lon_round", "date_round"]).copy()
print(f"6. After dedup (same ~300m pixel + day): {len(step6)}")

positives = step6[["Latitude", "Longitude", "event_date_parsed", "Location Accuracy",
                    "Landslide Trigger", "Imported Event Source Catalog"]].reset_index(drop=True)
positives = positives.rename(columns={"Latitude": "latitude", "Longitude": "longitude",
                                       "event_date_parsed": "event_date"})
positives["label"] = 1

print(f"\nFinal cleaned positives: {len(positives)}")
positives.head()


## 3. Joint spatio-temporal negative sampling -- HARD NEGATIVES (revised)

**Revision history:** the original version of this section sampled negatives from *anywhere* in the
2.2M km² ROI satisfying slope<5° / >250m from a positive / not water. That produced an elevation
bimodality in the negatives (stable terrain drawn from both low plains and high plateaus, while
positives cluster at low-to-mid elevation) which let Phase 4's models substantially learn
"is this flat/stable-sampled terrain" rather than genuine rainfall susceptibility -- see the Phase 4
"Known limitation" cell and Phase 5 Section 2 SHAP findings for the full diagnosis.

**Fix: each negative is now sampled from a local buffer around ONE specific positive**, not the whole
ROI. This keeps the negative in roughly the same regional elevation band, climate zone, and SAR
viewing geometry as the positive it's paired with, while still requiring it to be flat, dry, and
>250m from ANY positive (unchanged from before). The buffer radius widens adaptively (10 -> 20 -> 40
-> 80 km) if no stable candidate is found nearby, mirroring the same adaptive-window pattern already
used for spectral extraction in Phase 3 -- and like that pattern, the buffer size actually used is
recorded per negative (`neg_buffer_km`) for auditability rather than left invisible.

**Temporal matching:** each negative now inherits its paired positive's exact `event_date`, rather
than an independent draw from the global day-of-year/year distribution. This is a natural consequence
of the 1:1 spatial pairing and gives *exact* season-matching per pair instead of approximate
distribution-level matching.

In [ ]:
NEG_POS_RATIO = 1  # 1:1 by default; bump to 2 for 1:2 if the model needs more negatives later
LOCAL_BUFFER_STEPS_KM = [10, 20, 40, 80]  # adaptive widening -- same pattern as Phase 3's spectral windows
MIN_DIST_M = 250  # unchanged: exclude anything within this of ANY positive, not just the paired one
MAX_RETRIES = 4
BASE_BACKOFF_SECONDS = 2

import time

def call_with_retry(fn, *args, **kwargs):
    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            return fn(*args, **kwargs), None
        except Exception as e:
            last_error = str(e)[:300]
            time.sleep(BASE_BACKOFF_SECONDS * (2 ** attempt))
    return None, last_error


# --- Global stable-terrain mask (unchanged criteria: flat, dry, far from ANY positive) ---
positives_fc = ee.FeatureCollection(
    positives.apply(lambda r: ee.Feature(ee.Geometry.Point([r["longitude"], r["latitude"]])), axis=1).tolist()
)
dist_to_any_positive = positives_fc.distance(searchRadius=1000, maxError=30).unmask(9999)
far_enough_global = dist_to_any_positive.gt(MIN_DIST_M)

flat_terrain = srtm_slope.lt(5)
not_water = water_occurrence.lt(50)
stable_mask = flat_terrain.And(far_enough_global).And(not_water).selfMask()

print(f"Sampling hard negatives: one local buffer search per positive ({len(positives)} positives)...")


In [ ]:
neg_records = []
start_time = time.time()

for i, pos in positives.iterrows():
    pos_point = ee.Geometry.Point([pos["longitude"], pos["latitude"]])
    chosen = None
    used_buffer_km = None
    last_error = None

    for buf_km in LOCAL_BUFFER_STEPS_KM:
        local_region = pos_point.buffer(buf_km * 1000)

        def _sample(region=local_region, seed=42 + i):
            fc = stable_mask.sample(region=region, scale=90, numPixels=20,
                                     geometries=True, seed=seed, dropNulls=True, tileScale=4)
            return fc.getInfo()["features"]

        candidates, err = call_with_retry(_sample)
        if err is not None:
            last_error = err
            continue
        if candidates:
            rng_local = np.random.default_rng(seed=2000 + i)
            n_pick = min(NEG_POS_RATIO, len(candidates))
            picks = rng_local.choice(len(candidates), size=n_pick, replace=False)
            chosen = [candidates[p] for p in picks]
            used_buffer_km = buf_km
            break

    if chosen is None:
        neg_records.append({
            "pos_idx": i, "latitude": None, "longitude": None, "neg_buffer_km": None,
            "_neg_sample_error": last_error or f"no stable candidate within {max(LOCAL_BUFFER_STEPS_KM)}km"
        })
        continue

    for c in chosen:
        lon, lat = c["geometry"]["coordinates"]
        neg_records.append({"pos_idx": i, "latitude": lat, "longitude": lon, "neg_buffer_km": used_buffer_km})

    if (i + 1) % 20 == 0 or (i + 1) == len(positives):
        elapsed = time.time() - start_time
        print(f"  {i + 1}/{len(positives)} positives processed ({elapsed:.0f}s elapsed)")

neg_df = pd.DataFrame(neg_records)
n_failed = neg_df["latitude"].isna().sum()
print(f"\nHard-negative sampling: {len(neg_df) - n_failed} / {len(neg_df)} succeeded")
if n_failed:
    print(f"{n_failed} positive(s) had no stable-terrain candidate within {max(LOCAL_BUFFER_STEPS_KM)}km --")
    print("inspect the '_neg_sample_error' rows below; consider a larger max buffer step for these specifically")
    print("rather than silently dropping them, since each is a positive losing its paired negative.")
    print(neg_df[neg_df["latitude"].isna()][["pos_idx", "_neg_sample_error"]])

print("\nBuffer size actually used (auditability, same pattern as Phase 3's spectral_window_days):")
print(neg_df["neg_buffer_km"].value_counts().sort_index())

# --- Temporal matching: each negative inherits its paired positive's exact event_date (see markdown
# above for why this replaces the old global day-of-year/year bootstrap) ---
negatives = pd.DataFrame({
    "latitude": neg_df["latitude"],
    "longitude": neg_df["longitude"],
    "event_date": positives.loc[neg_df["pos_idx"], "event_date"].values,
    "neg_buffer_km": neg_df["neg_buffer_km"],
}).dropna(subset=["latitude", "longitude"]).reset_index(drop=True)

negatives["Location Accuracy"] = "N/A (sampled)"
negatives["Landslide Trigger"] = "N/A (negative)"
negatives["Imported Event Source Catalog"] = "N/A (sampled)"
negatives["label"] = 0

print(f"\nNegatives sampled (hard, elevation/context-matched): {len(negatives)}")
print(f"Positive:Negative ratio achieved: 1:{len(negatives) / len(positives):.2f}")
negatives.head()


## 4. Combine + assign spatial block ID
0.5°×0.5° grid cells, per the plan — but treated as a hypothesis to validate empirically below, not a final answer.

In [ ]:
combined = pd.concat([positives, negatives], ignore_index=True)

BLOCK_SIZE_DEG = 0.5
combined["block_id"] = (
    (combined["longitude"] // BLOCK_SIZE_DEG).astype(int).astype(str) + "_" +
    (combined["latitude"] // BLOCK_SIZE_DEG).astype(int).astype(str)
)

print(f"Combined dataset: {len(combined)} rows ({combined['label'].sum()} positive, "
      f"{(combined['label']==0).sum()} negative)")
print(f"Number of distinct spatial blocks: {combined['block_id'].nunique()}")
print("\nPositive-event dispersion across blocks (top 10):")
print(combined[combined["label"] == 1]["block_id"].value_counts().head(10))


## 5. Empirical spatial autocorrelation range check
Fits a spherical semivariogram on **elevation** (not slope) at the combined points to check whether 0.5° blocks are actually wide enough for spatially-independent folds. Slope was the original choice, but every negative sample is explicitly constrained to slope < 5° by the stable mask above — fitting a variogram on a variable your own sampling procedure constrains gives degenerate, near-zero variance and a meaningless (or crashing) range estimate. Elevation isn't constrained by the sampling design and has real spatial spread across this ROI. **Coordinates are projected before fitting** — raw lat/lon degrees would distort the range estimate given how much longitude compresses across 26–35°N. An azimuthal-equidistant projection centered on the ROI's own centroid is used instead of a single UTM zone, since the ROI spans too many degrees of longitude (74–97°E) for one UTM zone to stay accurate edge-to-edge.

In [ ]:
# Extract elevation at every combined point via sampleRegions(). Note: sampleRegions()
# silently DROPS points that fall on masked/no-data pixels rather than returning them
# with a null value -- so the output can be shorter than the input. We align results
# back to `combined` using the "idx" property (not positional order) to handle that.
combined_fc = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([row.longitude, row.latitude]), {"idx": i})
    for i, row in combined.iterrows()
])

elev_fc = srtm.select("elevation").sampleRegions(
    collection=combined_fc, scale=90, geometries=False, tileScale=4
)
elev_features = elev_fc.getInfo()["features"]

print("Raw sample of returned features (first 3):")
for f in elev_features[:3]:
    print(" ", f["properties"])
print(f"\nsampleRegions returned {len(elev_features)} / {len(combined)} points "
      f"({len(combined) - len(elev_features)} dropped -- fell on masked/no-data pixels).")

# Align by idx, not position, since sampleRegions can drop rows
elev_by_idx = {f["properties"]["idx"]: f["properties"].get("elevation", np.nan) for f in elev_features}
elev_values = combined.index.to_series().map(elev_by_idx).values
valid = ~np.isnan(elev_values)

print(f"Points with a valid elevation reading: {valid.sum()} / {len(elev_values)}")

if valid.sum() > 0:
    print(f"Elevation range among valid points: {np.nanmin(elev_values):.0f} to {np.nanmax(elev_values):.0f} m")
    print(f"Elevation std among valid points: {np.nanstd(elev_values[valid]):.1f} m")

if valid.sum() < 30:
    raise ValueError(
        "Too few points with a valid elevation reading to fit a variogram "
        f"({valid.sum()} valid). The printed raw features above should show whether "
        "the 'elevation' key is missing entirely, present but null, or something else -- "
        "use that to diagnose before continuing."
    )
if np.nanstd(elev_values[valid]) < 1.0:
    raise ValueError(
        "Elevation values are nearly constant across all points (std < 1m) -- "
        "a variogram can't be fit on data with no spatial variance. This would "
        "suggest a genuine data problem, not just a modeling choice -- investigate "
        "before trusting any block-size conclusion."
    )

# Project lon/lat -> meters using an azimuthal-equidistant projection centered on the ROI centroid
centroid_lon = (lon_min + lon_max) / 2
centroid_lat = (lat_min + lat_max) / 2
aeqd = pyproj.Proj(proj="aeqd", lat_0=centroid_lat, lon_0=centroid_lon, datum="WGS84", units="m")

x_m, y_m = aeqd(combined.loc[valid, "longitude"].values, combined.loc[valid, "latitude"].values)
coords_m = np.column_stack([x_m, y_m])

V = skg.Variogram(coords_m, elev_values[valid], model="spherical", maxlag="median", n_lags=25)
effective_range_m = V.parameters[0]

block_width_m = BLOCK_SIZE_DEG * 111_000  # rough deg->m at these latitudes, post-projection check
print(f"\nEmpirical spatial autocorrelation range (elevation): {effective_range_m/1000:.1f} km")
print(f"Current block width (0.5 deg): ~{block_width_m/1000:.1f} km")

if effective_range_m > block_width_m:
    print("\n⚠️  Autocorrelation range exceeds block width — blocks may be too small; "
          "neighboring blocks could still leak information across folds. Consider "
          "increasing BLOCK_SIZE_DEG or switching to spatial cluster K-fold.")
else:
    print("\n✅ Block width meets or exceeds the empirical autocorrelation range — "
          "0.5 deg blocking looks defensible for spatial K-fold.")


## 5b. Detrend elevation before trusting the range
The raw 815.5 km range is almost certainly inflated by elevation's large-scale regional trend (Gangetic plains → Himalayan peaks), not genuine local point-to-point autocorrelation. A variogram fit on a trending field keeps climbing with distance because of the trend itself, not local spatial structure — so before deciding block size, fit a 2nd-order polynomial trend surface and refit the variogram on the **residuals**, which isolate local variation.

In [ ]:
# Fit a 2nd-order polynomial trend surface: elevation ~ f(x, y)
x_valid, y_valid, z_valid = x_m, y_m, elev_values[valid]

A = np.column_stack([
    np.ones_like(x_valid), x_valid, y_valid,
    x_valid**2, y_valid**2, x_valid * y_valid
])
coeffs, *_ = np.linalg.lstsq(A, z_valid, rcond=None)
trend = A @ coeffs
residuals = z_valid - trend

r2 = 1 - np.sum(residuals**2) / np.sum((z_valid - z_valid.mean())**2)
print(f"Trend surface R^2: {r2:.3f}  (fraction of elevation variance explained by the large-scale trend)")
print(f"Residual std: {residuals.std():.1f} m  (vs raw elevation std: {z_valid.std():.1f} m)")

V_resid = skg.Variogram(coords_m, residuals, model="spherical", maxlag="median", n_lags=25)
effective_range_resid_m = V_resid.parameters[0]

print(f"\nDetrended (residual) spatial autocorrelation range: {effective_range_resid_m/1000:.1f} km")
print(f"Current block width (0.5 deg): ~{block_width_m/1000:.1f} km")

if effective_range_resid_m > block_width_m:
    recommended_deg = round(effective_range_resid_m / 111_000, 2)
    print(f"\n⚠️  Even after detrending, the local autocorrelation range "
          f"({effective_range_resid_m/1000:.1f} km) exceeds the 0.5 deg block width "
          f"({block_width_m/1000:.1f} km).")
    print(f"    Option A: increase BLOCK_SIZE_DEG to roughly {recommended_deg} deg.")
    print(f"    Option B (likely better given ROI size): switch to Spatial Cluster K-Fold "
          f"(e.g. KMeans on the projected coordinates) instead of a fixed grid -- a fixed "
          f"block size this large would leave very few distinct blocks across the ROI, "
          f"which starves K-fold of groups. This is the fallback the original plan "
          f"specifically called for in this situation.")
else:
    print("\n✅ After detrending, 0.5 deg blocking is defensible for spatial K-fold.")


## 6. Save cleaned dataset for Phase 3

In [ ]:
combined.to_csv("phase2_cleaned_dataset.csv", index=False)

print("Saved phase2_cleaned_dataset.csv")
print(f"\nFinal summary:")
print(f"  Total points: {len(combined)}")
print(f"  Positives: {combined['label'].sum()}")
print(f"  Negatives: {(combined['label']==0).sum()}")
print(f"  Spatial blocks: {combined['block_id'].nunique()}")
print(f"  Date range: {combined['event_date'].min()} to {combined['event_date'].max()}")


---
**Next: Phase 3 — spatio-temporal feature engineering.** Spectral indices, SAR backscatter/change metrics, rainfall thresholds, and topographic derivatives get extracted at each point in `phase2_cleaned_dataset.csv`, each source at its own native scale (not one shared `reduceRegion` scale across a stacked image).